# pyQuil Tutorial: The Quantum Computer

*Condensed from the [pyQuil docs, v4.17](https://pyquil-docs.rigetti.com/en/stable/the_quantum_computer.html).*

pyQuil is used to build Quil (Quantum Instruction Language) programs and execute them on simulated or real quantum processors. Quil is built on the idea that quantum computers will act as coprocessors alongside classical CPUs, running on a **Quantum Abstract Machine (QAM)** with a shared classical/quantum architecture.

A QAM implements abstract methods for manipulating classical and quantum state: loading programs, writing to shared memory, and executing programs.

Programs sent from pyQuil go to one of two endpoints:

- a **Quantum Virtual Machine (QVM)**
- a **Quantum Processing Unit (QPU)**

pyQuil provides `QVM` and `QPU` objects that talk to the APIs of those servers.

> **Prerequisites:** `pip install pyquil`. Cells that run programs need the `qvm` and `quilc` servers running (the setup cell below starts them). Cells marked `# Requires QCS access` need a Rigetti Quantum Cloud Services account and will fail on a local-only setup.

## Setup

Use the **Python (Rigetti)** kernel (see the README for one-time kernel registration). Run the next cell once per session to start the `quilc` and `qvm` servers in the background (`!cmd &` doesn't work in Jupyter, so this uses `subprocess.Popen`).

In [ ]:
import subprocess, shutil, sys, pyquil

for cmd in ("quilc", "qvm"):
    exe = shutil.which(cmd) or f"/ocean/projects/pscstaff/nnepal/Rigetti/.venv/bin/{cmd}"
    subprocess.Popen([exe, "-S"], stdout=open(f"{cmd}.log", "w"), stderr=subprocess.STDOUT)

print(sys.executable, pyquil.__version__)

## The Quantum Virtual Machine (QVM)

The QVM implements the Quantum Abstract Machine described in *A Practical Quantum Instruction Set Architecture*. It is written in ANSI Common Lisp and executes Quil programs. It ships with the Quil SDK and runs locally on your machine (see the QVM README or `qvm --help` for command-line usage).

pyQuil also has a **wavefunction simulator** for building and inspecting a program's wavefunction; see the next notebook.

## The Quantum Processing Unit (QPU)

To use a QPU you need a Quantum Cloud Services (QCS) account. Once authorized, you can submit jobs to a QPU from pyQuil, and your QCS dashboard lists the available processors.

## The `QuantumComputer`

`QuantumComputer` bundles the three objects you need to build and run programs:

- **`.qam`** — the quantum abstract machine, a `QVM` or `QPU` object
- **`.compiler`** — optimizes Quil and converts it into a form the QAM accepts
- **`.quantum_processor`** — the topology and Instruction Set Architecture (ISA): which 1-qubit and 2-qubit gates are supported

These subcomponents are always mutually compatible: a QPU-based `.qam` comes with a matching `QPUCompiler` and `.quantum_processor`.

### Instantiation

Create one with `get_qc()`:

```python
def get_qc(name: str, *, as_qvm: bool = None, noisy: bool = None, ...) -> QuantumComputer:
```

In [ ]:
from pyquil import get_qc

QPU_NAME = "Aspen-M-3"

# Get a QPU                                   # Requires QCS access
# qc = get_qc(QPU_NAME)

# Get a QVM with the same topology as the QPU  # Requires QCS access
# qc = get_qc(QPU_NAME, as_qvm=True)

# A fully connected QVM
number_of_qubits = 10
qc = get_qc(f"{number_of_qubits}q-qvm")
qc

> **Note:** You must join QCS to access specific quantum processors.
>
> **Note:** To connect to a local QVM, the QVM must be running in server mode (`qvm -S`).

### Methods

A typical workflow:

In [ ]:
from pyquil import get_qc, Program
from pyquil.gates import *

qc = get_qc('9q-square-qvm')

qubits = qc.qubits()
print(qubits)

p = Program()
# ... build program

compiled_program = qc.compile(p)

results = qc.run(compiled_program)

> **Note:** Both a QVM server and a `quilc` server must be running to compile and execute programs.

#### The `.run(...)` method

You are responsible for compiling your program before calling `.run()`:

In [ ]:
from pyquil import Program, get_qc
from pyquil.gates import X, MEASURE

qc = get_qc("8q-qvm")

p = Program()
ro = p.declare('ro', 'BIT', 2)
p += X(0)
p += MEASURE(0, ro[0])
p += MEASURE(1, ro[1])
p.wrap_in_numshots_loop(5)

executable = qc.compile(p)
result = qc.run(executable)
bitstrings = result.get_register_map().get("ro")
print(bitstrings)

Expected output:

```
[[1 0]
 [1 0]
 [1 0]
 [1 0]
 [1 0]]
```

The outer list enumerates trials (one per shot from `wrap_in_numshots_loop`). The inner list enumerates the values stored in the memory region `ro`, our readout register. Because of the `X` gate, qubit 0 always reads `1`.

> **Tip:** Extract qubit 0's results with `numpy.array(bitstrings)[:, 0]`.

In [ ]:
import numpy as np
print(np.array(bitstrings)[:, 0])

The result also carries execution metadata beyond the readout data, such as run duration — see `QAMExecutionResult`.

#### `.execute` and `.get_result`

`.run(...)` wraps two lower-level methods: `.execute(...)` and `.get_result(...)`. `.run()` looks synchronous, but some backends (such as live QPUs) run asynchronously. For finer control or concurrent execution, call these methods directly:

In [ ]:
job_handle = qc.qam.execute(executable)
result = qc.qam.get_result(job_handle)
print(result.get_register_map().get("ro"))

## Simulating the QPU using the QVM

`QAM` methods behave the same whether the target is a QVM or a QPU. As long as the topologies match, a program compiled and run on one will run on the other — which makes the QVM ideal for testing before using QPU time.

In [ ]:
# Requires QCS access
QPU_NAME = "Aspen-M-3"
qpu = get_qc(QPU_NAME)
qvm = get_qc(QPU_NAME, as_qvm=True)

With `as_qvm=True` you get a QVM with the same topology as the named QPU. For noise models, see the *Noise and quantum computation* notebook.

### Differences between a QVM- and a QPU-based `QuantumComputer`

For QVM targets, `qc.qam` is a `QVM` and `qc.compiler` is a `QVMCompiler`. For QPU targets, `qc.qam` is a `QPU` and `qc.compiler` is a `QPUCompiler`. They share common interfaces (`QAM` and `AbstractCompiler`), but each may have backend-specific methods. Use type checks to access those safely:

In [ ]:
from pyquil import get_qc
from pyquil.api import QPUCompiler

qc = get_qc("2q-qvm")  # or "Aspen-M-3"

if isinstance(qc.compiler, QPUCompiler):
    # Working with a QPU - refresh calibrations
    qc.compiler.get_calibration_program(force_refresh=True)
else:
    print("QVM target: no calibrations to refresh")

#### Requesting a job cancellation

While a job is pending on a QPU, you can cancel it with `cancel` (not available on the QVM). Check the QAM type first:

In [ ]:
from pyquil.api import QPU
from pyquil import Program
from pyquil.gates import X, MEASURE

p = Program()
ro = p.declare('ro', 'BIT', 1)
p += X(0)
p += MEASURE(0, ro[0])

job_handle = qc.qam.execute(qc.compile(p))

if isinstance(qc.qam, QPU):
    try:
        qc.qam.cancel(job_handle)
        print("Job was cancelled")
    except Exception:  # the docs catch QpuApiError, raised by the QCS SDK
        # If this error was raised, then the job failed to be cancelled.
        result = qc.qam.get_result(job_handle)
else:
    print("QVM target: cancellation not supported")

### Providing your own quantum processor topology

You can define a custom topology by specifying qubits and edges:

In [ ]:
# Requires QCS access (to read the Aspen-M-3 ISA)
import networkx as nx
from pyquil import get_qc
from pyquil.quantum_processor import NxQuantumProcessor
from pyquil.noise import decoherence_noise_with_asymmetric_ro

qpu = get_qc("Aspen-M-3")
isa = qpu.to_compiler_isa()
qubits = sorted(int(k) for k in isa.qubits.keys())[:16]
edges = [(q1, q2) for q1 in qubits for q2 in qubits if f"{q1}-{q2}" in isa.edges]

# Build the NX graph
topo = nx.from_edgelist(edges)
# You would uncomment the next line if you have disconnected qubits
# topo.add_nodes_from(qubits)
quantum_processor = NxQuantumProcessor(topo)
quantum_processor.noise_model = decoherence_noise_with_asymmetric_ro(quantum_processor.to_compiler_isa())  # Optional

A local-only variant that doesn't need QCS — a 4-qubit ring:

In [ ]:
import networkx as nx
from pyquil.quantum_processor import NxQuantumProcessor

ring = nx.cycle_graph(4)
ring_processor = NxQuantumProcessor(ring)
print(ring_processor.qubits())
print(ring_processor.edges())

With your custom processor, set `qc.compiler.quantum_processor` to it, or use it when creating new objects.